# MODULE 1
# Employee Health Data Management

installing the required libraires

In [306]:
%pip install pandas numpy scikit-learn joblib xgboost

Note: you may need to restart the kernel to use updated packages.


importing the libraires

In [307]:
import os
import pandas as pd
import numpy as np

loading the dataset

In [308]:
BASE_DIR = os.path.dirname(os.getcwd())

DATASET_PATH = os.path.join(BASE_DIR, "data", "dataset", "employee_wellness.csv")
PROCESSED_DATA_PATH = os.path.join(BASE_DIR, "data", "processed", "cleaned_employee_wellness.csv")
MODEL_PATH = os.path.join(BASE_DIR, "models", "wellness_risk_model.pkl")

read and show the first five rows of the dataset

In [309]:
df = pd.read_csv(DATASET_PATH)
df.head()

,employee_id,age,gender,height_cm,weight_kg,bmi,sleep_hours,exercise_days_per_week,attendance_percent,medical_condition,stress_score,smoker,alcohol_use,blood_pressure_systolic,blood_pressure_diastolic,glucose_level,risk_label
0,EMP101,34,Female,166.1,94.6,34.3,6.3,1,94.4,Stress-related fatigue,10.0,True,False,135,88,106.5,High
1,EMP102,23,Female,147.1,60.8,28.1,6.6,0,83.8,Mild fatigue,9.5,True,False,150,84,110.8,High
2,EMP103,29,Male,177.4,85.0,27.0,4.7,1,87.7,Hypertension risk,10.0,False,False,127,81,108.7,High
3,EMP104,57,Male,164.7,62.9,23.2,7.9,7,93.4,Mild fatigue,1.0,False,True,121,79,94.2,Low
4,EMP105,23,Male,168.1,62.4,22.1,8.5,5,99.7,Mild fatigue,1.8,False,False,125,75,81.9,Low


check for required columns if it is present

In [310]:
required_columns = [
    "employee_id",
    "age",
    "gender",
    "height_cm",
    "weight_kg",
    "bmi",
    "sleep_hours",
    "exercise_days_per_week",
    "attendance_percent",
    "medical_condition",
    "stress_score",
    "smoker",
    "alcohol_use",
    "blood_pressure_systolic",
    "blood_pressure_diastolic",
    "glucose_level",
    "risk_label"
]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    print("Missing columns:", missing_columns)
else:
    print("All required columns are present.")

All required columns are present.


Drop columns that exist but are effectively unusable

In [311]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 17 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   employee_id               30000 non-null  str    
 1   age                       30000 non-null  int64  
 2   gender                    30000 non-null  str    
 3   height_cm                 30000 non-null  float64
 4   weight_kg                 30000 non-null  float64
 5   bmi                       30000 non-null  float64
 6   sleep_hours               30000 non-null  float64
 7   exercise_days_per_week    30000 non-null  int64  
 8   attendance_percent        30000 non-null  float64
 9   medical_condition         30000 non-null  str    
 10  stress_score              30000 non-null  float64
 11  smoker                    30000 non-null  bool   
 12  alcohol_use               30000 non-null  bool   
 13  blood_pressure_systolic   30000 non-null  int64  
 14  blood_pressure_di

copy the dataset before cleaning

In [312]:
cleaned_df = df.copy()

numeric column conversion

In [313]:
numeric_columns = [
    "age",
    "height_cm",
    "weight_kg",
    "bmi",
    "sleep_hours",
    "exercise_days_per_week",
    "attendance_percent",
    "stress_score",
    "blood_pressure_systolic",
    "blood_pressure_diastolic",
    "glucose_level"
]

for col in numeric_columns:
    cleaned_df[col] = pd.to_numeric(cleaned_df[col], errors="coerce")
    cleaned_df[col] = cleaned_df[col].fillna(cleaned_df[col].median())

cleaning the catagorical columns and replacing the NaN and None

In [314]:
categorical_columns = [
    "gender",
    "medical_condition",
    "smoker",
    "alcohol_use",
    "risk_label"
]

for col in categorical_columns:
    cleaned_df[col] = cleaned_df[col].astype(str).str.strip()
    cleaned_df[col] = cleaned_df[col].replace(["nan", "None", ""], "Unknown")

In [315]:
# Convert smoker and alcohol_use to category type to ensure correct one-hot encoding\n",
cleaned_df['smoker'] = cleaned_df['smoker'].astype('category')
cleaned_df['alcohol_use'] = cleaned_df['alcohol_use'].astype('category')

show the number of columns and rows present in the dataset

In [316]:
cleaned_df = cleaned_df.drop_duplicates()
print("Shape after cleaning:", cleaned_df.shape)

Shape after cleaning: (30000, 17)


saving the processed dataset

In [317]:
cleaned_df.to_csv(PROCESSED_DATA_PATH, index=False)
print("Processed file saved at:", PROCESSED_DATA_PATH)

Processed file saved at: c:\Users\Lenovo\OneDrive\Documents\Infosys_Virtual_internship\Employee_Wellness_analytics\backend\data\processed\cleaned_employee_wellness.csv


# MODULE 2
# Wellness Risk Prediction

In [318]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
import joblib

In [319]:
model_df = pd.read_csv(PROCESSED_DATA_PATH)
model_df.head()

,employee_id,age,gender,height_cm,weight_kg,bmi,sleep_hours,exercise_days_per_week,attendance_percent,medical_condition,stress_score,smoker,alcohol_use,blood_pressure_systolic,blood_pressure_diastolic,glucose_level,risk_label
0,EMP101,34,Female,166.1,94.6,34.3,6.3,1,94.4,Stress-related fatigue,10.0,True,False,135,88,106.5,High
1,EMP102,23,Female,147.1,60.8,28.1,6.6,0,83.8,Mild fatigue,9.5,True,False,150,84,110.8,High
2,EMP103,29,Male,177.4,85.0,27.0,4.7,1,87.7,Hypertension risk,10.0,False,False,127,81,108.7,High
3,EMP104,57,Male,164.7,62.9,23.2,7.9,7,93.4,Mild fatigue,1.0,False,True,121,79,94.2,Low
4,EMP105,23,Male,168.1,62.4,22.1,8.5,5,99.7,Mild fatigue,1.8,False,False,125,75,81.9,Low


counting the values in the target column

In [320]:
print(model_df["risk_label"].value_counts())

risk_label
Low       11000
High       9500
Medium     9500
Name: count, dtype: int64


Encode target column

In [321]:
target_encoder = LabelEncoder()
model_df["risk_label_encoded"] = target_encoder.fit_transform(model_df["risk_label"])

print("Target classes:", list(target_encoder.classes_))

Target classes: ['High', 'Low', 'Medium']


In [322]:
feature_columns = [
    "age",
    "gender",
    "height_cm",
    "weight_kg",
    "bmi",
    "sleep_hours",
    "exercise_days_per_week",
    "attendance_percent",
    "stress_score",
    "medical_condition",
    "smoker",
    "alcohol_use",
    "blood_pressure_systolic",
    "blood_pressure_diastolic",
    "glucose_level"
]

X = model_df[feature_columns]
y = model_df["risk_label_encoded"]

One-hot encode categorical feature columns

In [323]:
#It converts the categorical columns in X into numeric 0/1 columns so the machine-learning model can use them.
X['smoker']= X['smoker'].astype('category')
X['alcohol_use']= X['alcohol_use'].astype('category')
X= pd.get_dummies(
    X,
    columns=["gender", "medical_condition", "smoker", "alcohol_use"],
    drop_first=True
)

print("Final feature shape:", X.shape)

Final feature shape: (30000, 18)


Train-test split

In [324]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

train random forest model

In [325]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstr

predict on test set

In [326]:
y_pred = rf_model.predict(X_test)

evaluate the model

In [327]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
print(classification_report(y_test, y_pred, target_names=target_encoder.classes_))


Accuracy: 0.9501666666666667
              precision    recall  f1-score   support

        High       0.97      0.96      0.96      1900
         Low       0.96      0.96      0.96      2200
      Medium       0.92      0.93      0.92      1900

    accuracy                           0.95      6000
   macro avg       0.95      0.95      0.95      6000
weighted avg       0.95      0.95      0.95      6000



save the trained model

In [328]:
joblib.dump(rf_model, MODEL_PATH)
joblib.dump(target_encoder, os.path.join(BASE_DIR, "models", "target_encoder.pkl"))
joblib.dump(X.columns.tolist(), os.path.join(BASE_DIR, "models", "feature_columns.pkl"))

print("Model and metadata saved successfully. Model saved at: ", MODEL_PATH)

print("Target encoder saved.")
print("Feature columns saved.")

Model and metadata saved successfully. Model saved at:  c:\Users\Lenovo\OneDrive\Documents\Infosys_Virtual_internship\Employee_Wellness_analytics\backend\models\wellness_risk_model.pkl
Target encoder saved.
Feature columns saved.


prediction on sample dataset

In [329]:
sample_input = {
    "age": 36,
    "gender": "Male",
    "heightCm": 170.6,
    "weightKg": 74.6,
    "bmi": 29.3,
    "sleepHoursPerNight": 7.6,
    "exerciseDaysPerWeek": 6,
    "attendanceRate": 95.7,
    "medicalCondition": "No major condition",
    "stressScore": 2.0,
    "smoker": False,
    "alcoholUse": False,
    "bloodPressureSystolic": 120,
    "bloodPressureDiastolic": 80,
    "glucoseLevel": 102.1
}

sample_df = pd.DataFrame([sample_input])

sample_df = pd.get_dummies(
    sample_df,
    columns=["gender", "medicalCondition", "smoker", "alcoholUse"],
    drop_first=True
)

saved_feature_columns = joblib.load(os.path.join(BASE_DIR, "models", "feature_columns.pkl"))
sample_df = sample_df.reindex(columns=saved_feature_columns, fill_value=0)

loaded_model = joblib.load(MODEL_PATH)
loaded_encoder = joblib.load(os.path.join(BASE_DIR, "models", "target_encoder.pkl"))

prediction = loaded_model.predict(sample_df)
predicted_label = loaded_encoder.inverse_transform(prediction)

print("Predicted risk label:", predicted_label[0])

Predicted risk label: Medium


# Module 3
# Recommendation System

In [330]:
%pip install cloudpickle

Note: you may need to restart the kernel to use updated packages.


In [362]:
# =====================================================================
# 1. DEFINE PURE PIPELINE FUNCTIONS (NO CLASS DEFINITION)
# =====================================================================
import os
import pandas as pd
import cloudpickle
from typing import Dict, Any, List

def _to_bool(value) -> bool:
    if isinstance(value, bool):
        return value
    if str(value).strip().lower() in ["true", "1", "yes"]:
        return True
    return False

def _get_profile_flags(employee: Dict[str, Any]) -> Dict[str, Any]:
    bmi = float(employee.get("bmi", employee.get("BMI", 24.0)))
    sleep = float(employee.get("sleepHoursPerNight", employee.get("sleep_hours", 7.0)))
    exercise = float(employee.get("exercise_days_per_week", employee.get("exerciseDaysPerWeek", 3.0)))
    stress = float(employee.get("stress_score", employee.get("stressScore", 5.0)))
    systolic = float(employee.get("blood_pressure_systolic", employee.get("bloodPressureSystolic", 120.0)))
    diastolic = float(employee.get("blood_pressure_diastolic", employee.get("bloodPressureDiastolic", 80.0)))
    glucose = float(employee.get("glucose_level", employee.get("glucoseLevel", 90.0)))
    attendance = float(employee.get("attendance_percent", employee.get("attendanceRate", 95.0)))
    
    medical_condition = str(employee.get("medical_condition", employee.get("medicalCondition", ""))).strip().lower()
    has_negation = any(neg in medical_condition for neg in ["no ", "none", "negative", "clear"])
    
    smoker = _to_bool(employee.get("smoker", False))
    alcohol = _to_bool(employee.get("alcohol_use", employee.get("alcoholUse", False)))
    risk = str(employee.get("risk_label", employee.get("riskType", "low"))).strip().lower()

    return {
        "high_bmi": bmi >= 30,
        "overweight": 25 <= bmi < 30,
        "low_sleep": sleep < 6,
        "borderline_sleep": 6 <= sleep < 7,
        "low_exercise": exercise <= 2,
        "moderate_exercise": 3 <= exercise <= 4,
        "high_stress": stress >= 8,
        "moderate_stress": 5 <= stress < 8,
        "high_bp": systolic >= 140 or diastolic >= 90,
        "borderline_bp": 130 <= systolic < 140 or 80 <= diastolic < 90,
        "high_glucose": glucose >= 126,
        "borderline_glucose": 100 <= glucose < 126,
        "low_attendance": attendance < 85,
        "fatigue": "fatigue" in medical_condition and not has_negation,
        "hypertension_risk": "hypertension" in medical_condition and not has_negation,
        "stress_related": "stress" in medical_condition and not has_negation,
        "smoker": smoker,
        "alcohol_use": alcohol,
        "high_risk": risk == "high",
        "medium_risk": risk == "medium",
        "low_risk": risk == "low"
    }

def _score_single_recommendation(employee: Dict[str, Any], rec: pd.Series) -> Dict[str, Any]:
    flags = _get_profile_flags(employee)
    score = 0.0
    reasons = []

    category = str(rec["category"]).strip().lower()
    tags = [t.strip().lower() for t in str(rec["target_tags"]).split("|") if t.strip()]
    risk_levels = [r.strip().lower() for r in str(rec["risk_levels"]).split("|") if r.strip()]

    if flags["high_risk"] and "high" in risk_levels:
        score += 3.0
        reasons.append("Matches high-risk wellness profile")
    elif flags["medium_risk"] and "medium" in risk_levels:
        score += 3.0
        reasons.append("Matches medium-risk wellness profile")
    elif flags["low_risk"] and "low" in risk_levels:
        score += 3.0
        reasons.append("Matches low-risk wellness profile")

    if "weight_loss" in tags and flags["high_bmi"]:
        score += 4.0
        reasons.append("BMI indicates obesity-focused support")
    elif "weight_loss" in tags and flags["overweight"]:
        score += 2.0
        reasons.append("BMI indicates weight-management support")

    if "sleep_improvement" in tags and flags["low_sleep"]:
        score += 4.0
        reasons.append("Sleep hours are below healthy range")
    elif "sleep_improvement" in tags and flags["borderline_sleep"]:
        score += 2.0
        reasons.append("Sleep routine can be improved")

    if "stress_relief" in tags and flags["high_stress"]:
        score += 4.0
        reasons.append("Stress score is very high")
    elif "stress_relief" in tags and flags["moderate_stress"]:
        score += 2.0
        reasons.append("Stress score is moderately elevated")

    if "activity_boost" in tags and flags["low_exercise"]:
        score += 4.0
        reasons.append("Exercise frequency is low")
    elif "activity_boost" in tags and flags["moderate_exercise"]:
        score += 1.0
        reasons.append("Exercise can be improved further")

    if "bp_control" in tags and flags["high_bp"]:
        score += 4.0
        reasons.append("Blood pressure is elevated")
    elif "bp_control" in tags and flags["borderline_bp"]:
        score += 2.0
        reasons.append("Blood pressure is borderline")

    if "glucose_control" in tags and flags["high_glucose"]:
        score += 4.0
        reasons.append("Glucose level is high")
    elif "glucose_control" in tags and flags["borderline_glucose"]:
        score += 2.0
        reasons.append("Glucose level is borderline")

    if "fatigue_recovery" in tags and flags["fatigue"]:
        score += 3.0
        reasons.append("Medical record notes chronic fatigue strain")
    if "stress_relief" in tags and flags["stress_related"]:
        score += 3.0
        reasons.append("Medical history points to stress symptoms")
    if "bp_control" in tags and flags["hypertension_risk"]:
        score += 3.0
        reasons.append("Medical record tracks hypertension history")

    if "smoking_cessation" in tags and flags["smoker"]:
        score += 4.0
        reasons.append("Smoking habit intervention active")
    if "alcohol_moderation" in tags and flags["alcohol_use"]:
        score += 3.0
        reasons.append("Alcohol reduction goals apply")
    if "recovery_support" in tags and flags["low_attendance"]:
        score += 2.0
        reasons.append("Attendance pattern reflects personal health strains")

    if category == "fitness" and (flags["high_risk"] or flags["high_bp"]):
        score -= 3.0
        reasons.append("High-intensity physical plans moderated for risk control")

    priority_val = float(rec.get("priority_score", 0))
    score += priority_val

    return {
        "recommendation_id": rec["recommendation_id"],
        "title": rec["title"],
        "category": rec["category"],
        "description": rec["description"],
        "score": round(score, 2),
        "priority_score": priority_val,
        "reasons": reasons[:3]
    }

def _get_wellness_recommendations(employee: Dict[str, Any], recommendations_df: pd.DataFrame, top_n: int = 5) -> List[Dict[str, Any]]:
    scored = [_score_single_recommendation(employee, rec) for _, rec in recommendations_df.iterrows()]
    
    # 1. Deduplicate recommendations by title (stripping whitespace to prevent hidden mismatches)
    unique_by_title = {}
    for rec in sorted(scored, key=lambda x: x['score'], reverse=True):
        clean_title = str(rec['title']).strip()
        if clean_title not in unique_by_title:
            unique_by_title[clean_title] = rec
    
    # 2. Promote diversity by slightly boosting items from different categories
    diverse_recs = []
    seen_categories = set()
    remaining_recs = list(unique_by_title.values())

    while len(diverse_recs) < top_n and remaining_recs:
        remaining_recs.sort(
            key=lambda x: (x['score'] + (2 if x['category'] not in seen_categories else 0), x['priority_score']), 
            reverse=True
        )
        best_rec = remaining_recs.pop(0)
        diverse_recs.append(best_rec)
        seen_categories.add(best_rec['category'])

    return [rec for rec in diverse_recs if rec["score"] > 0][:top_n]

# =====================================================================
# 2. DYNAMIC PATH RESOLUTION & DEDUPLICATION
# =====================================================================
try:
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    current_dir = os.getcwd()

if os.path.basename(current_dir).lower() in ["notebook", "notebooks"]:
    BASE_DIR = os.path.dirname(current_dir)
else:
    BASE_DIR = current_dir

print(f"🎯 Project Base Directory Set To: {BASE_DIR}")
csv_path = os.path.join(BASE_DIR, "data", "dataset", "wellness_catalog.csv")

recommendations_df = pd.read_csv(csv_path).drop_duplicates(subset=["recommendation_id"])
recommendations_df.to_csv(csv_path, index=False)
print("✅ Deduplicated dataset safely written back to disk.")

# =====================================================================
# 3. SAVE THE FUNCTION OVER THE PICKLE FILE (CRITICAL STEP)
# =====================================================================
def recommendation_engine(employee: Dict[str, Any], top_n: int = 5) -> List[Dict[str, Any]]:
    return _get_wellness_recommendations(employee, recommendations_df, top_n)

pickle_out_path = os.path.join(BASE_DIR, "models", "wellness_recommendation_engine.pkl")

with open(pickle_out_path, "wb") as f:
    cloudpickle.dump(recommendation_engine, f)

print(f"📦 Successfully overwrote engine file with diversity features: {pickle_out_path}")

🎯 Project Base Directory Set To: c:\Users\Lenovo\OneDrive\Documents\Infosys_Virtual_internship\Employee_Wellness_analytics\backend
✅ Deduplicated dataset safely written back to disk.
📦 Successfully overwrote engine file with diversity features: c:\Users\Lenovo\OneDrive\Documents\Infosys_Virtual_internship\Employee_Wellness_analytics\backend\models\wellness_recommendation_engine.pkl


In [361]:
# ==========================================
# 4. LIVE VERIFICATION RUN (TESTING SUITE)
# ==========================================
import cloudpickle  # Make sure cloudpickle is imported to read the function logic

print("🔄 Running confirmation test suite on freshly baked model...")

# FIX 1: Safely load the pure functional closure file using cloudpickle
with open(RECOMMENDATION_MODEL_PATH, "rb") as f:
    loaded_engine = cloudpickle.load(f)

test_profiles = [
    {
        "scenario_name": "CASE 1: High Stress & Burnout Profile (Medium Risk)",
        "employee_name": "Ananya Sharma",
        "bmi": 24.2,
        "sleepHoursPerNight": 5.1,
        "exercise_days_per_week": 1,
        "stress_score": 8.5,
        "blood_pressure_systolic": 122,
        "blood_pressure_diastolic": 81,
        "glucose_level": 92.0,
        "attendance_percent": 82.5,
        "medical_condition": "Chronic stress-related fatigue symptoms",
        "smoker": False,
        "alcohol_use": False,
        "risk_label": "Medium"
    },
    {
        "scenario_name": "CASE 2: Cardiovascular & Metabolic Risks (High Risk Safeguard)",
        "employee_name": "David Miller",
        "bmi": 34.5,
        "sleepHoursPerNight": 6.2,
        "exercise_days_per_week": 0,
        "stress_score": 4.0,
        "blood_pressure_systolic": 145,
        "blood_pressure_diastolic": 92,
        "glucose_level": 132.0,
        "attendance_percent": 96.0,
        "medical_condition": "Diagnosed Hypertension history",
        "smoker": True,
        "alcohol_use": True,
        "risk_label": "High"
    }
]

for profile in test_profiles:
    print(f"📌 {profile['scenario_name']}")
    print(f"👤 target: {profile['employee_name']} | ML Risk Classification: {profile['risk_label']}")
    print("-" * 50)
    
    # FIX 2: Call the loaded function directly! No more object dot-notation methods.
    recs = loaded_engine(profile, top_n=3)
    
    for idx, r in enumerate(recs, 1):
        print(f"   {idx}. [{r['category'].upper()}] {r['title']} (Score: {r['score']})")
        print(f"      Reasons: {', '.join(r['reasons'])}")
    print("\n" + "-"*50)

🔄 Running confirmation test suite on freshly baked model...
📌 CASE 1: High Stress & Burnout Profile (Medium Risk)
👤 target: Ananya Sharma | ML Risk Classification: Medium
--------------------------------------------------
   1. [YOGA] Desk Yoga and Stretching (Score: 18.0)
      Reasons: Matches medium-risk wellness profile, Stress score is very high, Exercise frequency is low
   2. [YOGA] Desk Yoga and Stretching (Score: 18.0)
      Reasons: Matches medium-risk wellness profile, Stress score is very high, Exercise frequency is low
   3. [YOGA] Desk Yoga and Stretching (Score: 18.0)
      Reasons: Matches medium-risk wellness profile, Stress score is very high, Exercise frequency is low

--------------------------------------------------
📌 CASE 2: Cardiovascular & Metabolic Risks (High Risk Safeguard)
👤 target: David Miller | ML Risk Classification: High
--------------------------------------------------
   1. [LIFESTYLE] Smoking Cessation Coaching (Score: 16.0)
      Reasons: Matches 

example recomendation

# Module 4
Mental Health and Sentimental Analysis

Install required libraries

In [333]:
%pip install pandas numpy nltk scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.


Import libraries

In [334]:
import os
import re
import joblib
import numpy as np
import pandas as pd

import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

Download NLTK resources

In [335]:
nltk.download("vader_lexicon")
nltk.download("punkt")
nltk.download("stopwords")

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

Set project paths

In [336]:
import os

BASE_DIR = os.getcwd()
PROJECT_ROOT = os.path.dirname(BASE_DIR)

RAW_DATA_PATH = os.path.join(PROJECT_ROOT, "data", "dataset", "employee_feedback.csv")
PROCESSED_DATA_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "sentiment_results.csv")
DEPT_SUMMARY_PATH = os.path.join(PROJECT_ROOT, "outputs", "department_stress_summary.csv")
ALERT_DEPTS_PATH = os.path.join(PROJECT_ROOT, "outputs", "top_alert_departments.csv")
MODEL_PATH = os.path.join(PROJECT_ROOT, "models", "sentiment_pipeline.pkl")

Load the anonymized pulse-check data

In [337]:
df = pd.read_csv(RAW_DATA_PATH)
df.head()

,feedback_id,employee_id,department,feedback_date,feedback_text,rating,sentiment
0,FB101,EMP101,Operations,2024-10-19,"Just doing my daily tasks, things are normal.",3,Neutral
1,FB102,EMP102,Customer Support,2025-05-17,Collaborative environment and friendly colleag...,4,Positive
2,FB103,EMP103,Operations,2025-02-19,Collaborative environment and friendly colleag...,5,Positive
3,FB104,EMP104,HR,2025-01-30,Lack of career progression and training.,2,Negative
4,FB105,EMP105,Engineering,2025-08-19,"Management is alright, some good days and bad ...",3,Neutral


Validate required columns

In [338]:
required_columns = ["feedback_id", "department", "feedback_text", "feedback_date"]
missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    print("Missing columns:", missing_columns)
else:
    print("All required columns are present.")

All required columns are present.


Clean and preprocess text

In [339]:
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)

    try:
        tokens = word_tokenize(text)
    except LookupError:
        tokens = text.split()

    tokens = [word for word in tokens if word not in stop_words]
    return " ".join(tokens)

df["cleaned_feedback"] = df["feedback_text"].astype(str).apply(clean_text)
df[["feedback_text", "cleaned_feedback"]].head()

,feedback_text,cleaned_feedback
0,"Just doing my daily tasks, things are normal.",daily tasks things normal
1,Collaborative environment and friendly colleag...,collaborative environment friendly colleagues
2,Collaborative environment and friendly colleag...,collaborative environment friendly colleagues
3,Lack of career progression and training.,lack career progression training
4,"Management is alright, some good days and bad ...",management alright good days bad days


Add sentiment scoring

In [340]:
sia = SentimentIntensityAnalyzer()

def get_sentiment_scores(text):
    scores = sia.polarity_scores(str(text))
    return pd.Series({
        "neg": scores["neg"],
        "neu": scores["neu"],
        "pos": scores["pos"],
        "compound": scores["compound"]
    })

sentiment_scores = df["feedback_text"].apply(get_sentiment_scores)
df = pd.concat([df, sentiment_scores], axis=1)

df.head()

,feedback_id,employee_id,department,feedback_date,feedback_text,rating,sentiment,cleaned_feedback,neg,neu,pos,compound
0,FB101,EMP101,Operations,2024-10-19,"Just doing my daily tasks, things are normal.",3,Neutral,daily tasks things normal,0.000,1.000,0.000,0.0000
1,FB102,EMP102,Customer Support,2025-05-17,Collaborative environment and friendly colleag...,4,Positive,collaborative environment friendly colleagues,0.000,0.556,0.444,0.4939
2,FB103,EMP103,Operations,2025-02-19,Collaborative environment and friendly colleag...,5,Positive,collaborative environment friendly colleagues,0.000,0.556,0.444,0.4939
3,FB104,EMP104,HR,2025-01-30,Lack of career progression and training.,2,Negative,lack career progression training,0.315,0.685,0.000,-0.3182
4,FB105,EMP105,Engineering,2025-08-19,"Management is alright, some good days and bad ...",3,Neutral,management alright good days bad days,0.243,0.417,0.340,0.1027


Create sentiment labels

In [341]:
def classify_sentiment(compound_score):
    if compound_score >= 0.05:
        return "Positive"
    elif compound_score <= -0.05:
        return "Negative"
    return "Neutral"

df["sentiment_label"] = df["compound"].apply(classify_sentiment)
df[["feedback_text", "compound", "sentiment_label"]].head()

,feedback_text,compound,sentiment_label
0,"Just doing my daily tasks, things are normal.",0.0000,Neutral
1,Collaborative environment and friendly colleag...,0.4939,Positive
2,Collaborative environment and friendly colleag...,0.4939,Positive
3,Lack of career progression and training.,-0.3182,Negative
4,"Management is alright, some good days and bad ...",0.1027,Positive


Detect stress, burnout, and anxiety keywords

In [342]:
stress_keywords = [
    "stress", "stressed", "pressure", "overwhelmed", "burnout",
    "burned out", "exhausted", "anxious", "anxiety", "fatigue",
    "tired", "drained", "workload", "deadline", "panic"
]

def detect_stress_terms(text):
    text = str(text).lower()
    matched = [word for word in stress_keywords if word in text]
    return pd.Series({
        "stress_term_count": len(matched),
        "stress_terms_found": ", ".join(matched)
    })

stress_info = df["feedback_text"].apply(detect_stress_terms)
df = pd.concat([df, stress_info], axis=1)

df.head()

,feedback_id,employee_id,department,feedback_date,feedback_text,rating,sentiment,cleaned_feedback,neg,neu,pos,compound,sentiment_label,stress_term_count,stress_terms_found
0,FB101,EMP101,Operations,2024-10-19,"Just doing my daily tasks, things are normal.",3,Neutral,daily tasks things normal,0.000,1.000,0.000,0.0000,Neutral,0,
1,FB102,EMP102,Customer Support,2025-05-17,Collaborative environment and friendly colleag...,4,Positive,collaborative environment friendly colleagues,0.000,0.556,0.444,0.4939,Positive,0,
2,FB103,EMP103,Operations,2025-02-19,Collaborative environment and friendly colleag...,5,Positive,collaborative environment friendly colleagues,0.000,0.556,0.444,0.4939,Positive,0,
3,FB104,EMP104,HR,2025-01-30,Lack of career progression and training.,2,Negative,lack career progression training,0.315,0.685,0.000,-0.3182,Negative,0,
4,FB105,EMP105,Engineering,2025-08-19,"Management is alright, some good days and bad ...",3,Neutral,management alright good days bad days,0.243,0.417,0.340,0.1027,Positive,0,


Build a stress risk flag

In [343]:
def classify_stress_risk(row):
    if row["stress_term_count"] >= 2 or row["compound"] <= -0.4:
        return "High"
    elif row["stress_term_count"] == 1 or row["compound"] < 0:
        return "Medium"
    return "Low"

df["stress_risk"] = df.apply(classify_stress_risk, axis=1)
df[["feedback_text", "sentiment_label", "stress_term_count", "stress_risk"]].head()

,feedback_text,sentiment_label,stress_term_count,stress_risk
0,"Just doing my daily tasks, things are normal.",Neutral,0,Low
1,Collaborative environment and friendly colleag...,Positive,0,Low
2,Collaborative environment and friendly colleag...,Positive,0,Low
3,Lack of career progression and training.,Negative,0,Medium
4,"Management is alright, some good days and bad ...",Positive,0,Low


Aggregate by department

In [344]:
department_summary = df.groupby("department").agg(
    total_feedback=("feedback_id", "count"),
    avg_compound_sentiment=("compound", "mean"),
    negative_feedback_count=("sentiment_label", lambda x: (x == "Negative").sum()),
    high_stress_count=("stress_risk", lambda x: (x == "High").sum()),
    medium_stress_count=("stress_risk", lambda x: (x == "Medium").sum())
).reset_index()

department_summary["negative_feedback_pct"] = (
    department_summary["negative_feedback_count"] / department_summary["total_feedback"] * 100
).round(2)

department_summary["high_stress_pct"] = (
    department_summary["high_stress_count"] / department_summary["total_feedback"] * 100
).round(2)

department_summary = department_summary.sort_values(
    by=["high_stress_pct", "negative_feedback_pct"],
    ascending=False
)

department_summary

,department,total_feedback,avg_compound_sentiment,negative_feedback_count,high_stress_count,medium_stress_count,negative_feedback_pct,high_stress_pct
1,Engineering,3716,0.187611,873,631,413,23.49,16.98
4,IT,3773,0.188222,921,609,459,24.41,16.14
0,Customer Support,3711,0.185574,912,597,468,24.58,16.09
7,Sales,3849,0.193069,924,606,464,24.01,15.74
5,Marketing,3735,0.206506,843,583,402,22.57,15.61
3,HR,3757,0.207134,859,561,434,22.86,14.93
2,Finance,3674,0.200632,850,546,436,23.14,14.86
6,Operations,3785,0.198201,872,561,450,23.04,14.82


Create an alert list for departments

In [345]:
top_alert_departments = department_summary[
    (department_summary["high_stress_pct"] >= 30) |
    (department_summary["avg_compound_sentiment"] <= -0.20)
].copy()

top_alert_departments

,department,total_feedback,avg_compound_sentiment,negative_feedback_count,high_stress_count,medium_stress_count,negative_feedback_pct,high_stress_pct


Save the processed files

In [346]:
df.to_csv(PROCESSED_DATA_PATH, index=False)
department_summary.to_csv(DEPT_SUMMARY_PATH, index=False)
top_alert_departments.to_csv(ALERT_DEPTS_PATH, index=False)

print("Processed feedback saved to:", PROCESSED_DATA_PATH)
print("Department summary saved to:", DEPT_SUMMARY_PATH)
print("Alert departments saved to:", ALERT_DEPTS_PATH)

Processed feedback saved to: c:\Users\Lenovo\OneDrive\Documents\Infosys_Virtual_internship\Employee_Wellness_analytics\backend\data\processed\sentiment_results.csv
Department summary saved to: c:\Users\Lenovo\OneDrive\Documents\Infosys_Virtual_internship\Employee_Wellness_analytics\backend\outputs\department_stress_summary.csv
Alert departments saved to: c:\Users\Lenovo\OneDrive\Documents\Infosys_Virtual_internship\Employee_Wellness_analytics\backend\outputs\top_alert_departments.csv


Save a reusable text pipeline

In [347]:
class TextCleaner(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return [clean_text(text) for text in X]

sentiment_pipeline = Pipeline([
    ("cleaner", TextCleaner()),
    ("tfidf", TfidfVectorizer(max_features=500))
])

sentiment_pipeline.fit(df["feedback_text"])

joblib.dump(sentiment_pipeline, MODEL_PATH)
print("Pipeline saved to:", MODEL_PATH)

Pipeline saved to: c:\Users\Lenovo\OneDrive\Documents\Infosys_Virtual_internship\Employee_Wellness_analytics\backend\models\sentiment_pipeline.pkl


Load the saved pipeline later

In [348]:
loaded_pipeline = joblib.load(MODEL_PATH)
sample_vectors = loaded_pipeline.transform([
    "I am feeling overloaded and mentally exhausted due to deadlines."
])

print("Loaded pipeline works. Shape:", sample_vectors.shape)

Loaded pipeline works. Shape: (1, 80)


testing

In [349]:
test_feedback = pd.DataFrame({
    "feedback_text": [
        "I feel supported and positive about my work.",
        "The pressure is too high and I am exhausted.",
        "Work is okay but deadlines are stressful."
    ]
})

test_scores = test_feedback["feedback_text"].apply(get_sentiment_scores)
test_feedback = pd.concat([test_feedback, test_scores], axis=1)
test_feedback["sentiment_label"] = test_feedback["compound"].apply(classify_sentiment)

test_stress = test_feedback["feedback_text"].apply(detect_stress_terms)
test_feedback = pd.concat([test_feedback, test_stress], axis=1)
test_feedback["stress_risk"] = test_feedback.apply(classify_stress_risk, axis=1)

test_feedback

,feedback_text,neg,neu,pos,compound,sentiment_label,stress_term_count,stress_terms_found,stress_risk
0,I feel supported and positive about my work.,0.000,0.459,0.541,0.7096,Positive,0,,Low
1,The pressure is too high and I am exhausted.,0.439,0.561,0.000,-0.5719,Negative,2,"pressure, exhausted",High
2,Work is okay but deadlines are stressful.,0.408,0.459,0.133,-0.6124,Negative,2,"stress, deadline",High
